# 🤖 Student Performance — Machine Learning

**Epsilon AI Machine Learning Program**  
**Dataset:** `student_performance_grade.csv` — 8,000 students × 19 features  
**Tasks:** Grade Classification + Final Score Regression

---

## 📑 Table of Contents
1. [Data Preparation](#section-1)
2. [Preprocessing Pipeline](#section-2)
3. [Classification — Grade Prediction](#section-3)
4. [Regression — Final Score Prediction](#section-4)
5. [Hyperparameter Tuning](#section-5)
6. [Validation & Generalization](#section-6)
7. [Final Model Comparison](#section-7)

---

In [ ]:
# ============================================================
# IMPORTS — All ML libraries
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import pickle

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.spines.top': False, 'axes.spines.right': False})

PALETTE       = ['#0D9488', '#14B8A6', '#5EEAD4', '#0F766E', '#F59E0B', '#EF4444']
GRADE_ORDER   = ['A', 'B', 'C', 'D', 'Fail']
GRADE_PALETTE = {'A': '#0D9488', 'B': '#14B8A6', 'C': '#F59E0B', 'D': '#F97316', 'Fail': '#EF4444'}

print("✅ Libraries imported")

In [ ]:
# ============================================================
# SKLEARN & XGBOOST IMPORTS
# ============================================================

# Preprocessing
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold)
from sklearn.preprocessing  import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.impute          import SimpleImputer

# Classification models
from sklearn.linear_model    import LogisticRegression, LinearRegression
from sklearn.ensemble        import RandomForestClassifier, RandomForestRegressor
from xgboost                 import XGBClassifier, XGBRegressor

# Evaluation — classification
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
# Evaluation — regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Model & metric imports complete")

<a id='section-1'></a>
# 📁 Section 1: Data Preparation

---

We load the feature-engineered dataset produced by `EDA_Analysis.ipynb` and prepare our feature matrix and both target vectors.

---

In [ ]:
# ============================================================
# LOAD FEATURE-ENGINEERED DATASET
# ============================================================

try:
    df = pd.read_csv('student_performance_fe.csv')
    print("✅ Loaded: student_performance_fe.csv")
except FileNotFoundError:
    print("⚠️  Running EDA pipeline inline...")
    df = pd.read_csv('student_performance_grade.csv')
    # Winsorize
    for col in ['Hours_Studied','Attendance','Sleep_Hours','Screen_Time',
                'Previous_GPA','Exam_Anxiety_Score','Final_Score']:
        Q1,Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        df[col] = df[col].clip(Q1-1.5*(Q3-Q1), Q3+1.5*(Q3-Q1))
    # Engineer
    extra_flag = (df['Extracurricular']=='Yes').astype(int)
    df['Study_Efficiency']    = df['Hours_Studied']*(df['Attendance']/100)
    df['Academic_Risk_Index'] = (df['Stress_Level']+df['Exam_Anxiety_Score']+df['Screen_Time'])/3 - df['Previous_GPA']*2
    df['Engagement_Index']    = df['Attendance']*0.4 + df['Tutoring_Sessions_Per_Week']*10 + extra_flag*8 + df['Hours_Studied']*3
    df['Wellness_Score']      = df['Sleep_Hours']*10 - df['Stress_Level']*5 - df['Exam_Anxiety_Score']*3
    df['Learning_Efficiency'] = df['Previous_GPA']*df['Hours_Studied']/(1+df['Screen_Time'])
    print("✅ Dataset prepared inline")

print(f"\nShape: {df.shape}")
df.head()

In [ ]:
# ============================================================
# DEFINE FEATURE GROUPS
# ============================================================

numerical_features = [
    'Age', 'Hours_Studied', 'Attendance', 'Sleep_Hours', 'Stress_Level',
    'Screen_Time', 'Previous_GPA', 'Tutoring_Sessions_Per_Week',
    'Exam_Anxiety_Score', 'Study_Efficiency', 'Academic_Risk_Index',
    'Engagement_Index', 'Wellness_Score', 'Learning_Efficiency'
]

binary_features = ['Part_Time_Job', 'Extracurricular']

ordinal_features = ['Diet_Quality', 'Internet_Quality', 'Family_Income_Level']
ordinal_categories = [
    ['Poor', 'Average', 'Good'],
    ['Poor', 'Average', 'Good', 'Excellent'],
    ['Low', 'Middle', 'High']
]

nominal_features = ['Gender', 'Study_Method']

all_features = numerical_features + binary_features + ordinal_features + nominal_features

# Feature matrix
X = df[all_features].copy()

# Targets
y_class = df['Grade']       # Classification target
y_reg   = df['Final_Score'] # Regression target

print(f"✅ Feature matrix X:   {X.shape}")
print(f"✅ Classification target — Grade:       {y_class.nunique()} classes → {sorted(y_class.unique())}")
print(f"✅ Regression target   — Final_Score:   range {y_reg.min():.1f} – {y_reg.max():.1f}")
print(f"\nFeature Breakdown:")
print(f"  Numerical  ({len(numerical_features)}): {numerical_features}")
print(f"  Binary     ({len(binary_features)}):  {binary_features}")
print(f"  Ordinal    ({len(ordinal_features)}):  {ordinal_features}")
print(f"  Nominal    ({len(nominal_features)}):  {nominal_features}")

In [ ]:
# ============================================================
# GRADE DISTRIBUTION IN TARGET
# ============================================================

grade_counts = y_class.value_counts().reindex(GRADE_ORDER)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [GRADE_PALETTE[g] for g in GRADE_ORDER]
bars = ax.bar(GRADE_ORDER, grade_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, grade_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Target Distribution — Grade Classes', fontsize=13, fontweight='bold')
ax.set_xlabel('Grade')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print("\n📝 Class Imbalance Note:")
print(f"  Majority class (A): {grade_counts['A']:,} ({grade_counts['A']/len(df)*100:.1f}%)")
print(f"  Minority class (Fail): {grade_counts['Fail']:,} ({grade_counts['Fail']/len(df)*100:.1f}%)")
print("  → Stratified split will ensure proportional class representation in train/test.")

<a id='section-2'></a>
# 🔧 Section 2: Preprocessing Pipeline

---

A scikit-learn `Pipeline` + `ColumnTransformer` ensures:
- **No data leakage** — transformations are fit only on training data
- **Reproducibility** — same transformation applied to any new data
- **Clean code** — single object handles all preprocessing

### Pipeline Architecture

```
X (raw)
│
├── Numerical   →  SimpleImputer(median)  →  StandardScaler
├── Binary      →  SimpleImputer(mode)    →  OrdinalEncoder
├── Ordinal     →  SimpleImputer(mode)    →  OrdinalEncoder (ordered)
└── Nominal     →  SimpleImputer(mode)    →  OrdinalEncoder
```

---

In [ ]:
# ============================================================
# PREPROCESSING PIPELINES — EACH FEATURE GROUP
# ============================================================

# 1. Numerical: impute missing → scale to mean=0, std=1
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# 2. Binary: impute mode → encode as 0/1
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# 3. Ordinal: impute mode → encode preserving natural order
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value', unknown_value=-1
    ))
])

# 4. Nominal: impute mode → encode (no meaningful order)
nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# 5. Combined ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_features),
    ('bin', binary_pipeline,    binary_features),
    ('ord', ordinal_pipeline,   ordinal_features),
    ('nom', nominal_pipeline,   nominal_features),
], remainder='drop')

print("✅ Preprocessing Pipeline Built:")
print("   Numerical   →  Median Imputation + StandardScaler")
print("   Binary      →  Mode Imputation + OrdinalEncoder (0/1)")
print("   Ordinal     →  Mode Imputation + OrdinalEncoder (ordered)")
print("   Nominal     →  Mode Imputation + OrdinalEncoder")

In [ ]:
# ============================================================
# TRAIN-TEST SPLIT — 80/20 STRATIFIED
# ============================================================

# Encode labels for XGBoost compatibility
le = LabelEncoder()
y_class_enc = le.fit_transform(y_class)
grade_names  = le.classes_   # ['A', 'B', 'C', 'D', 'Fail']

# Classification split (stratified)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class_enc, test_size=0.2, random_state=42, stratify=y_class_enc
)

# Regression split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print("✅ Train-Test Split (80% / 20%):")
print(f"   Classification — Train: {X_train_c.shape[0]:,}  |  Test: {X_test_c.shape[0]:,}")
print(f"   Regression     — Train: {X_train_r.shape[0]:,}  |  Test: {X_test_r.shape[0]:,}")

print("\n📊 Class distribution in test set:")
test_dist = pd.Series(le.inverse_transform(y_test_c)).value_counts().reindex(GRADE_ORDER)
for grade, cnt in test_dist.items():
    pct = cnt / len(y_test_c) * 100
    print(f"   {grade:<6} {cnt:>5} ({pct:.1f}%)")

<a id='section-3'></a>
# 🎯 Section 3: Classification — Grade Prediction

---

**Target:** `Grade` ∈ {A, B, C, D, Fail} — 5-class classification

We train and compare **3 models**:
1. **Logistic Regression** — linear baseline
2. **Random Forest Classifier** — bagging ensemble
3. **XGBoost Classifier** — gradient boosting

---

In [ ]:
# ============================================================
# HELPER — CLASSIFICATION EVALUATION FUNCTION
# ============================================================

def evaluate_classifier(pipeline, X_test, y_test, model_name, grade_names):
    """
    Full evaluation of a classification model.
    Returns metrics dict and plots confusion matrix.
    """
    y_pred = pipeline.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred,    average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred,        average='weighted', zero_division=0)

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=grade_names, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
                xticklabels=grade_names, yticklabels=grade_names, ax=ax,
                linewidths=0.5, linecolor='white')
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Grade', fontsize=11)
    ax.set_ylabel('Actual Grade', fontsize=11)
    plt.tight_layout()
    plt.show()

    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1}

clf_results = []
clf_models  = {}
print("✅ Evaluation function ready")

## 3.1 Model 1 — Logistic Regression (Baseline)

**What it is:** A linear model that estimates class probabilities using the logistic (sigmoid) function extended to multiple classes via a softmax layer.

**Why baseline:** Fast, interpretable, and provides a lower bound. If complex models don't significantly beat it, the data may be linearly separable or too small.

**Limitation:** Cannot capture non-linear feature interactions.

In [ ]:
# ============================================================
# CLASSIFICATION MODEL 1 — LOGISTIC REGRESSION
# ============================================================

lr_clf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        max_iter=2000,
        solver='lbfgs',
        multi_class='auto',
        C=1.0,
        random_state=42
    ))
])

print("Training Logistic Regression...")
lr_clf_pipe.fit(X_train_c, y_train_c)
print("✅ Training complete")

metrics_lr_clf = evaluate_classifier(
    lr_clf_pipe, X_test_c, y_test_c,
    'Logistic Regression', grade_names
)
clf_results.append(metrics_lr_clf)
clf_models['Logistic Regression'] = lr_clf_pipe

## 3.2 Model 2 — Random Forest Classifier

**What it is:** An ensemble of `n_estimators` decision trees, each trained on a bootstrap sample with random feature subsets. Final prediction = majority vote.

**Advantage:** Handles non-linearity and feature interactions; provides built-in feature importance; robust to noise.

**Limitation:** Less interpretable than Logistic Regression; slower training.

In [ ]:
# ============================================================
# CLASSIFICATION MODEL 2 — RANDOM FOREST
# ============================================================

rf_clf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training Random Forest Classifier...")
rf_clf_pipe.fit(X_train_c, y_train_c)
print("✅ Training complete")

metrics_rf_clf = evaluate_classifier(
    rf_clf_pipe, X_test_c, y_test_c,
    'Random Forest Classifier', grade_names
)
clf_results.append(metrics_rf_clf)
clf_models['Random Forest Classifier'] = rf_clf_pipe

## 3.3 Model 3 — XGBoost Classifier

**What it is:** Gradient Boosted Trees — each tree sequentially corrects the errors (residuals) of the previous one, minimizing a differentiable loss function (multi-class log loss).

**Advantage:** State-of-the-art on tabular data; built-in L1/L2 regularization; handles class imbalance; fast with GPU/parallelism.

**Limitation:** More hyperparameters to tune; can overfit without regularization.

In [ ]:
# ============================================================
# CLASSIFICATION MODEL 3 — XGBOOST
# ============================================================

xgb_clf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42,
        verbosity=0
    ))
])

print("Training XGBoost Classifier...")
xgb_clf_pipe.fit(X_train_c, y_train_c)
print("✅ Training complete")

metrics_xgb_clf = evaluate_classifier(
    xgb_clf_pipe, X_test_c, y_test_c,
    'XGBoost Classifier', grade_names
)
clf_results.append(metrics_xgb_clf)
clf_models['XGBoost Classifier'] = xgb_clf_pipe

In [ ]:
# ============================================================
# CLASSIFICATION — COMPARISON TABLE
# ============================================================

clf_df = pd.DataFrame(clf_results).set_index('Model')

print("\n" + "="*65)
print("   CLASSIFICATION MODELS — COMPARISON TABLE")
print("="*65)
print(clf_df.round(4).to_string())
print()

best_clf = clf_df['F1 Score'].idxmax()
print(f"🏆 Best Classification Model: {best_clf}")
print(f"   F1 Score: {clf_df.loc[best_clf, 'F1 Score']:.4f}")
print(f"   Accuracy: {clf_df.loc[best_clf, 'Accuracy']:.4f}")

In [ ]:
# ============================================================
# CLASSIFICATION — COMPARISON BAR CHART
# ============================================================

fig, ax = plt.subplots(figsize=(13, 6))

x      = np.arange(len(clf_df.index))
width  = 0.2
metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors_plot  = ['#0D9488', '#14B8A6', '#F59E0B', '#0F766E']

for i, (metric, color) in enumerate(zip(metrics_plot, colors_plot)):
    bars = ax.bar(x + i*width, clf_df[metric], width, label=metric,
                  color=color, edgecolor='white', linewidth=1.2)
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{bar.get_height():.3f}', ha='center', fontsize=8, rotation=90, va='bottom')

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(clf_df.index, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0.0, 1.15)
ax.set_title('Classification Models — Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.4, linewidth=1, label='90% threshold')

plt.tight_layout()
plt.show()

<a id='section-4'></a>
# 📈 Section 4: Regression — Final Score Prediction

---

**Target:** `Final_Score` (continuous, range 22.81 – 99.98)

We train and compare **3 models**:
1. **Linear Regression** — linear baseline
2. **Random Forest Regressor** — bagging ensemble
3. **XGBoost Regressor** — gradient boosting

**Metrics:** MAE · MSE · RMSE · R²

---

In [ ]:
# ============================================================
# HELPER — REGRESSION EVALUATION FUNCTION
# ============================================================

def evaluate_regressor(pipeline, X_test, y_test, model_name):
    """
    Full evaluation of a regression model.
    Returns metrics dict and plots actual vs predicted + residuals.
    """
    y_pred = pipeline.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test,  y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  MAE  (Mean Absolute Error):        {mae:.4f} points")
    print(f"  MSE  (Mean Squared Error):         {mse:.4f}")
    print(f"  RMSE (Root Mean Squared Error):    {rmse:.4f} points")
    print(f"  R²   (Coefficient of Determination): {r2:.4f}")
    print(f"  → Model explains {r2*100:.1f}% of variance in Final Score")

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Actual vs Predicted
    axes[0].scatter(y_test, y_pred, alpha=0.3, s=15, color='#0D9488')
    lo = min(y_test.min(), y_pred.min()); hi = max(y_test.max(), y_pred.max())
    axes[0].plot([lo, hi], [lo, hi], 'r--', linewidth=2, label='Perfect fit')
    axes[0].set_title(f'{model_name}\nActual vs Predicted', fontweight='bold')
    axes[0].set_xlabel('Actual Final Score')
    axes[0].set_ylabel('Predicted Final Score')
    axes[0].legend()
    axes[0].text(0.05, 0.95, f'R² = {r2:.3f}', transform=axes[0].transAxes,
                 fontsize=11, fontweight='bold', va='top', color='#0D9488')

    # Residuals vs Predicted
    residuals = y_test.values - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.3, s=15, color='#F59E0B')
    axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
    axes[1].set_title(f'{model_name}\nResiduals Plot', fontweight='bold')
    axes[1].set_xlabel('Predicted Values')
    axes[1].set_ylabel('Residuals (Actual − Predicted)')

    # Residuals distribution
    axes[2].hist(residuals, bins=50, color='#0D9488', edgecolor='white', alpha=0.85, density=True)
    from scipy.stats import norm as norm_dist
    mu, sigma = residuals.mean(), residuals.std()
    x_norm = np.linspace(residuals.min(), residuals.max(), 200)
    axes[2].plot(x_norm, norm_dist.pdf(x_norm, mu, sigma), 'r-', linewidth=2, label='Normal fit')
    axes[2].set_title(f'{model_name}\nResiduals Distribution', fontweight='bold')
    axes[2].set_xlabel('Residuals')
    axes[2].set_ylabel('Density')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    return {'Model': model_name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2}

reg_results = []
reg_models  = {}
print("✅ Regression evaluation function ready")

## 4.1 Model 1 — Linear Regression (Baseline)

**What it is:** Fits a hyperplane that minimizes the sum of squared residuals between predictions and actual values. Each coefficient directly represents the marginal effect of a feature.

**Limitation:** Assumes linearity between features and target. Cannot model non-linear patterns or feature interactions.

In [ ]:
# ============================================================
# REGRESSION MODEL 1 — LINEAR REGRESSION
# ============================================================

lr_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

print("Training Linear Regression...")
lr_reg_pipe.fit(X_train_r, y_train_r)
print("✅ Training complete")

metrics_lr_reg = evaluate_regressor(lr_reg_pipe, X_test_r, y_test_r, 'Linear Regression')
reg_results.append(metrics_lr_reg)
reg_models['Linear Regression'] = lr_reg_pipe

## 4.2 Model 2 — Random Forest Regressor

**What it is:** Grows `n_estimators` trees on bootstrap samples; each tree predicts a score; final output is the **average** of all tree predictions. Reduces variance without increasing bias (bagging effect).

**Advantage:** Handles non-linear relationships, feature interactions; less prone to overfitting than single trees.

In [ ]:
# ============================================================
# REGRESSION MODEL 2 — RANDOM FOREST REGRESSOR
# ============================================================

rf_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training Random Forest Regressor...")
rf_reg_pipe.fit(X_train_r, y_train_r)
print("✅ Training complete")

metrics_rf_reg = evaluate_regressor(rf_reg_pipe, X_test_r, y_test_r, 'Random Forest Regressor')
reg_results.append(metrics_rf_reg)
reg_models['Random Forest Regressor'] = rf_reg_pipe

## 4.3 Model 3 — XGBoost Regressor

**What it is:** Gradient boosting minimizes MSE loss function sequentially. Each new tree is fit on the negative gradient (residuals) of the loss function from the previous ensemble.

**Advantage:** Often best performance on tabular data; L1/L2 regularization (lambda/alpha); native handling of missing values; fast with parallel tree construction.

In [ ]:
# ============================================================
# REGRESSION MODEL 3 — XGBOOST REGRESSOR
# ============================================================

xgb_reg_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    ))
])

print("Training XGBoost Regressor...")
xgb_reg_pipe.fit(X_train_r, y_train_r)
print("✅ Training complete")

metrics_xgb_reg = evaluate_regressor(xgb_reg_pipe, X_test_r, y_test_r, 'XGBoost Regressor')
reg_results.append(metrics_xgb_reg)
reg_models['XGBoost Regressor'] = xgb_reg_pipe

In [ ]:
# ============================================================
# REGRESSION — COMPARISON TABLE
# ============================================================

reg_df = pd.DataFrame(reg_results).set_index('Model')

print("\n" + "="*65)
print("   REGRESSION MODELS — COMPARISON TABLE")
print("="*65)
print(reg_df.round(4).to_string())

best_reg = reg_df['R²'].idxmax()
print(f"\n🏆 Best Regression Model: {best_reg}")
print(f"   R²:   {reg_df.loc[best_reg, 'R²']:.4f}")
print(f"   MAE:  {reg_df.loc[best_reg, 'MAE']:.4f} points")
print(f"   RMSE: {reg_df.loc[best_reg, 'RMSE']:.4f} points")

In [ ]:
# ============================================================
# REGRESSION — COMPARISON CHARTS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

metrics_reg = ['MAE', 'RMSE', 'R²']
colors_reg  = ['#EF4444', '#F97316', '#0D9488']

for i, (metric, color) in enumerate(zip(metrics_reg, colors_reg)):
    vals = reg_df[metric]
    bars = axes[i].bar(reg_df.index, vals, color=['#0D9488','#14B8A6','#F59E0B'], edgecolor='white')
    for bar in bars:
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                     f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')
    axes[i].set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)

    # Add "lower is better" / "higher is better" note
    note = '(↓ Lower is better)' if metric in ['MAE','RMSE'] else '(↑ Higher is better)'
    axes[i].set_xlabel(note, fontsize=9, color='gray')

plt.suptitle('Regression Models — Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-5'></a>
# 🔬 Section 5: Hyperparameter Tuning

---

## What Are Hyperparameters?

Hyperparameters are **configuration settings** that are NOT learned from data — they must be specified before training begins. Examples:

| Hyperparameter | Description |
|----------------|-------------|
| `n_estimators` | Number of trees in the ensemble |
| `max_depth` | Maximum depth of each tree |
| `learning_rate` | Step size for gradient updates (XGBoost) |
| `min_samples_split` | Minimum samples needed to split a node |

## Why is Tuning Important?

Default values rarely achieve optimal performance. Tuning can:
- Reduce overfitting (e.g., limiting `max_depth`)
- Improve generalization on unseen data
- Squeeze out additional performance on competitive datasets

## Method: GridSearchCV

`GridSearchCV` exhaustively tries every combination from a parameter grid using **k-fold cross-validation** to evaluate each. The best combination maximizes the chosen metric.

---

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING — RANDOM FOREST CLASSIFIER
# ============================================================

print("Tuning Random Forest Classifier...")
print("Parameter grid: n_estimators × max_depth × min_samples_split")
print()

rf_param_grid_clf = {
    'model__n_estimators':    [100, 200, 300],
    'model__max_depth':       [None, 10, 20, 30],
    'model__min_samples_split': [2, 5]
}

rf_tuned_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_grid_clf = GridSearchCV(
    rf_tuned_clf, rf_param_grid_clf,
    cv=5, scoring='f1_weighted',
    n_jobs=-1, verbose=1
)

rf_grid_clf.fit(X_train_c, y_train_c)

print(f"\n✅ Best Parameters: {rf_grid_clf.best_params_}")
print(f"   Best CV F1 Score: {rf_grid_clf.best_score_:.4f}")

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING — XGBOOST CLASSIFIER
# ============================================================

print("Tuning XGBoost Classifier...")

xgb_param_grid_clf = {
    'model__n_estimators':  [100, 200, 300],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__max_depth':     [4, 6, 8]
}

xgb_tuned_clf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        use_label_encoder=False, eval_metric='mlogloss',
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    ))
])

xgb_grid_clf = GridSearchCV(
    xgb_tuned_clf, xgb_param_grid_clf,
    cv=5, scoring='f1_weighted',
    n_jobs=-1, verbose=1
)

xgb_grid_clf.fit(X_train_c, y_train_c)

print(f"\n✅ Best Parameters: {xgb_grid_clf.best_params_}")
print(f"   Best CV F1 Score: {xgb_grid_clf.best_score_:.4f}")

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING — RANDOM FOREST REGRESSOR
# ============================================================

print("Tuning Random Forest Regressor...")

rf_param_grid_reg = {
    'model__n_estimators':    [100, 200, 300],
    'model__max_depth':       [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

rf_tuned_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_grid_reg = GridSearchCV(
    rf_tuned_reg, rf_param_grid_reg,
    cv=5, scoring='r2',
    n_jobs=-1, verbose=1
)

rf_grid_reg.fit(X_train_r, y_train_r)

print(f"\n✅ Best Parameters: {rf_grid_reg.best_params_}")
print(f"   Best CV R² Score: {rf_grid_reg.best_score_:.4f}")

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING — XGBOOST REGRESSOR
# ============================================================

print("Tuning XGBoost Regressor...")

xgb_param_grid_reg = {
    'model__n_estimators':  [100, 200, 300],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__max_depth':     [4, 6, 8]
}

xgb_tuned_reg = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    ))
])

xgb_grid_reg = GridSearchCV(
    xgb_tuned_reg, xgb_param_grid_reg,
    cv=5, scoring='r2',
    n_jobs=-1, verbose=1
)

xgb_grid_reg.fit(X_train_r, y_train_r)

print(f"\n✅ Best Parameters: {xgb_grid_reg.best_params_}")
print(f"   Best CV R² Score: {xgb_grid_reg.best_score_:.4f}")

In [ ]:
# ============================================================
# TUNING RESULTS — BEFORE vs AFTER COMPARISON
# ============================================================

print("\n" + "="*70)
print("   HYPERPARAMETER TUNING — BEFORE vs AFTER COMPARISON")
print("="*70)

# Classification
before_rf_f1  = f1_score(y_test_c, clf_models['Random Forest Classifier'].predict(X_test_c), average='weighted', zero_division=0)
after_rf_f1   = f1_score(y_test_c, rf_grid_clf.best_estimator_.predict(X_test_c),            average='weighted', zero_division=0)
before_xgb_f1 = f1_score(y_test_c, clf_models['XGBoost Classifier'].predict(X_test_c),       average='weighted', zero_division=0)
after_xgb_f1  = f1_score(y_test_c, xgb_grid_clf.best_estimator_.predict(X_test_c),           average='weighted', zero_division=0)

print("\nClassification (F1 Weighted):")
print(f"  Random Forest  — Before: {before_rf_f1:.4f}  |  After: {after_rf_f1:.4f}  |  Δ = {after_rf_f1-before_rf_f1:+.4f}")
print(f"  XGBoost        — Before: {before_xgb_f1:.4f}  |  After: {after_xgb_f1:.4f}  |  Δ = {after_xgb_f1-before_xgb_f1:+.4f}")

# Regression
before_rf_r2  = r2_score(y_test_r, reg_models['Random Forest Regressor'].predict(X_test_r))
after_rf_r2   = r2_score(y_test_r, rf_grid_reg.best_estimator_.predict(X_test_r))
before_xgb_r2 = r2_score(y_test_r, reg_models['XGBoost Regressor'].predict(X_test_r))
after_xgb_r2  = r2_score(y_test_r, xgb_grid_reg.best_estimator_.predict(X_test_r))

print("\nRegression (R²):")
print(f"  Random Forest  — Before: {before_rf_r2:.4f}  |  After: {after_rf_r2:.4f}  |  Δ = {after_rf_r2-before_rf_r2:+.4f}")
print(f"  XGBoost        — Before: {before_xgb_r2:.4f}  |  After: {after_xgb_r2:.4f}  |  Δ = {after_xgb_r2-before_xgb_r2:+.4f}")

In [ ]:
# ============================================================
# BEFORE vs AFTER — VISUALIZATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models_tuned = ['RF Classifier', 'XGB Classifier']
before_clf = [before_rf_f1, before_xgb_f1]
after_clf  = [after_rf_f1,  after_xgb_f1]

x = np.arange(len(models_tuned))
axes[0].bar(x - 0.2, before_clf, 0.35, label='Before Tuning', color='#5EEAD4', edgecolor='white')
axes[0].bar(x + 0.2, after_clf,  0.35, label='After Tuning',  color='#0D9488', edgecolor='white')
for i in range(len(models_tuned)):
    axes[0].text(i-0.2, before_clf[i]+0.002, f'{before_clf[i]:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[0].text(i+0.2, after_clf[i]+0.002,  f'{after_clf[i]:.3f}',  ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(models_tuned)
axes[0].set_ylabel('F1 Score (Weighted)'); axes[0].set_ylim(0.90, 1.0)
axes[0].set_title('Classification: Before vs After Tuning', fontweight='bold')
axes[0].legend()

models_tuned_r = ['RF Regressor', 'XGB Regressor']
before_reg = [before_rf_r2, before_xgb_r2]
after_reg  = [after_rf_r2,  after_xgb_r2]

x = np.arange(len(models_tuned_r))
axes[1].bar(x - 0.2, before_reg, 0.35, label='Before Tuning', color='#FCD34D', edgecolor='white')
axes[1].bar(x + 0.2, after_reg,  0.35, label='After Tuning',  color='#F59E0B', edgecolor='white')
for i in range(len(models_tuned_r)):
    axes[1].text(i-0.2, before_reg[i]+0.001, f'{before_reg[i]:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[1].text(i+0.2, after_reg[i]+0.001,  f'{after_reg[i]:.3f}',  ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(models_tuned_r)
axes[1].set_ylabel('R² Score'); axes[1].set_ylim(0.90, 1.0)
axes[1].set_title('Regression: Before vs After Tuning', fontweight='bold')
axes[1].legend()

plt.suptitle('Hyperparameter Tuning — Before vs After Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-6'></a>
# ✅ Section 6: Validation & Generalization

---

## Key Concepts

| Concept | Definition | How to Detect |
|---------|-----------|---------------|
| **Overfitting** | High train score, low test score | Large train−test gap |
| **Underfitting** | Low score on both sets | Baseline barely beats |
| **Generalization** | Consistent performance on unseen data | Small train−test gap |
| **Cross-Validation** | k-fold rotation of train/val splits | Low CV variance |

---

In [ ]:
# ============================================================
# CROSS-VALIDATION — 5-FOLD STRATIFIED (CLASSIFICATION)
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results_clf = {}
print("5-Fold Cross-Validation — Classification (F1 Weighted):")
print("-" * 55)
for name, pipeline in clf_models.items():
    scores = cross_val_score(pipeline, X, y_class_enc, cv=cv,
                              scoring='f1_weighted', n_jobs=-1)
    cv_results_clf[name] = scores
    print(f"  {name:<30} {scores.mean():.4f} ± {scores.std():.4f}")
    print(f"  {'':30} Individual folds: {np.round(scores, 4)}")
    print()

In [ ]:
# ============================================================
# TRAIN vs TEST SCORE COMPARISON (OVERFITTING CHECK)
# ============================================================

print("Overfitting Check — Train Score vs Test Score:")
print("-" * 65)
for name, pipeline in clf_models.items():
    train_f1 = f1_score(y_train_c, pipeline.predict(X_train_c), average='weighted', zero_division=0)
    test_f1  = f1_score(y_test_c,  pipeline.predict(X_test_c),  average='weighted', zero_division=0)
    gap      = train_f1 - test_f1
    status   = "✅ Good" if gap < 0.05 else "⚠️ Overfit" if gap > 0.1 else "🟡 Mild"
    print(f"  {name:<30}  Train: {train_f1:.4f}  Test: {test_f1:.4f}  Gap: {gap:+.4f}  {status}")

In [ ]:
# ============================================================
# CROSS-VALIDATION BOXPLOT — CLASSIFICATION
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
positions = range(len(cv_results_clf))
for i, (name, scores) in enumerate(cv_results_clf.items()):
    bp = ax.boxplot(scores, positions=[i], widths=0.45, patch_artist=True,
                    boxprops=dict(facecolor=PALETTE[i], alpha=0.7),
                    medianprops=dict(color='black', linewidth=2.5),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

ax.set_xticks(list(positions))
ax.set_xticklabels(cv_results_clf.keys(), fontsize=11)
ax.set_ylabel('F1 Score (Weighted)')
ax.set_title('5-Fold Cross-Validation — Classification Models', fontsize=13, fontweight='bold')
ax.axhline(y=cv_results_clf[list(cv_results_clf.keys())[-1]].mean(),
           color='red', linestyle='--', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()

print("\n📝 Narrow boxplots = low variance = strong generalization")
print("   Tall/wide boxplots = high variance = potential overfitting")

In [ ]:
# ============================================================
# CROSS-VALIDATION — REGRESSION (R²)
# ============================================================

print("5-Fold Cross-Validation — Regression (R²):")
print("-" * 55)
for name, pipeline in reg_models.items():
    scores = cross_val_score(pipeline, X, y_reg, cv=5,
                              scoring='r2', n_jobs=-1)
    train_r2 = r2_score(y_train_r, pipeline.predict(X_train_r))
    test_r2  = r2_score(y_test_r,  pipeline.predict(X_test_r))
    gap      = train_r2 - test_r2
    print(f"  {name:<30}  CV: {scores.mean():.4f} ± {scores.std():.4f}  "
          f"Train: {train_r2:.4f}  Test: {test_r2:.4f}  Gap: {gap:+.4f}")

<a id='section-7'></a>
# 🏆 Section 7: Final Model Comparison & Selection

---

We compile all results into final comparison tables, select the best models, and save them for use in the Streamlit application.

---

In [ ]:
# ============================================================
# FINAL CLASSIFICATION RESULTS TABLE
# ============================================================

print("\n" + "="*70)
print("   FINAL CLASSIFICATION COMPARISON — ALL MODELS")
print("="*70)
print(clf_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(clf_df.index))
width = 0.2
for i, (metric, color) in enumerate(zip(['Accuracy','Precision','Recall','F1 Score'],
                                          ['#0D9488','#14B8A6','#F59E0B','#0F766E'])):
    ax.bar(x + i*width, clf_df[metric], width, label=metric, color=color, edgecolor='white')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(clf_df.index, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('🏆 Final Classification Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

best_clf_name = clf_df['F1 Score'].idxmax()
print(f"\n🏆 BEST CLASSIFICATION MODEL: {best_clf_name}")
print(f"   Accuracy:  {clf_df.loc[best_clf_name, 'Accuracy']:.4f}")
print(f"   Precision: {clf_df.loc[best_clf_name, 'Precision']:.4f}")
print(f"   Recall:    {clf_df.loc[best_clf_name, 'Recall']:.4f}")
print(f"   F1 Score:  {clf_df.loc[best_clf_name, 'F1 Score']:.4f}")

In [ ]:
# ============================================================
# FINAL REGRESSION RESULTS TABLE
# ============================================================

print("\n" + "="*70)
print("   FINAL REGRESSION COMPARISON — ALL MODELS")
print("="*70)
print(reg_df[['MAE','RMSE','R²']].round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for i, (metric, title) in enumerate([('MAE','↓ Lower = Better'),
                                       ('RMSE','↓ Lower = Better'),
                                       ('R²','↑ Higher = Better')]):
    colors_b = ['#0D9488','#14B8A6','#F59E0B']
    bars = axes[i].bar(reg_df.index, reg_df[metric], color=colors_b, edgecolor='white')
    for bar in bars:
        axes[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                     f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')
    axes[i].set_title(f'{metric} — {title}', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=15)
plt.suptitle('🏆 Final Regression Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_reg_name = reg_df['R²'].idxmax()
print(f"\n🏆 BEST REGRESSION MODEL: {best_reg_name}")
print(f"   R²:   {reg_df.loc[best_reg_name, 'R²']:.4f}")
print(f"   MAE:  {reg_df.loc[best_reg_name, 'MAE']:.4f} points")
print(f"   RMSE: {reg_df.loc[best_reg_name, 'RMSE']:.4f} points")

In [ ]:
# ============================================================
# SAVE BEST MODELS
# ============================================================

best_clf_model = clf_models[best_clf_name]
best_reg_model = reg_models[best_reg_name]

with open('best_clf_model.pkl', 'wb') as f:
    pickle.dump(best_clf_model, f)

with open('best_reg_model.pkl', 'wb') as f:
    pickle.dump(best_reg_model, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("💾 Models saved:")
print(f"   best_clf_model.pkl  → {best_clf_name}")
print(f"   best_reg_model.pkl  → {best_reg_name}")
print(f"   label_encoder.pkl   → Grade label encoder")

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*70)
print("   PROJECT ML SUMMARY — STUDENT PERFORMANCE")
print("="*70)
print(f"  Dataset:         8,000 students × 19 features")
print(f"  Engineered:      5 new features")
print(f"  Selected:        12 most informative features")
print()
print(f"  🎯 CLASSIFICATION TASK (Grade Prediction)")
print(f"     Best Model:  {best_clf_name}")
print(f"     Accuracy:    {clf_df.loc[best_clf_name, 'Accuracy']*100:.1f}%")
print(f"     F1 Score:    {clf_df.loc[best_clf_name, 'F1 Score']*100:.1f}%")
print()
print(f"  📈 REGRESSION TASK (Final Score Prediction)")
print(f"     Best Model:  {best_reg_name}")
print(f"     R² Score:    {reg_df.loc[best_reg_name, 'R²']*100:.1f}%")
print(f"     RMSE:        {reg_df.loc[best_reg_name, 'RMSE']:.2f} points")
print()
print("  ✅ Both models saved and ready for Streamlit deployment!")